In [5]:
# The code is divided to multiple parts:
# 1. Pre-processing
# 2. Model architecture
# 3. Training
# 4. WebCam demo
# 5. Image accuracy test for model analysis
# 6. Video visualiser
# 7. Image visualiser

In [1]:
# Pre-processing

In [8]:
import os
import shutil
import cv2
import glob
import numpy as np
import xml.etree.ElementTree as ET

dataset_root = "/Users/shudhanshuranjangupta/human/Train"
output_root = "processed_data_64x128"
IMG_SIZE = (64, 128)

if os.path.exists(output_root):
    shutil.rmtree(output_root)

os.makedirs(f"{output_root}/1_human", exist_ok=True)
os.makedirs(f"{output_root}/0_background", exist_ok=True)

image_paths = glob.glob(os.path.join(dataset_root, "JPEGImages", "*.png"))

def check_overlap(c, humans):
    for h in humans:
        if not (c[2] < h[0] or c[0] > h[2] or c[3] < h[1] or c[1] > h[3]): 
            return True
    return False

for img_path in image_paths:
    file_id = os.path.splitext(os.path.basename(img_path))[0]
    xml_path = os.path.join(dataset_root, "Annotations", file_id + ".xml")
    
    img = cv2.imread(img_path)
    if img is None: continue
    h_img, w_img = img.shape[:2]

    human_boxes = []
    if os.path.exists(xml_path):
        root = ET.parse(xml_path).getroot()
        for obj in root.findall('object'):
            name = obj.find('name').text.lower()
            if 'person' in name or 'pedestrian' in name:
                bnd = obj.find('bndbox')
                human_boxes.append([
                    int(float(bnd.find('xmin').text)),
                    int(float(bnd.find('ymin').text)),
                    int(float(bnd.find('xmax').text)),
                    int(float(bnd.find('ymax').text))
                ])

    if human_boxes:
        for i, (x1, y1, x2, y2) in enumerate(human_boxes):
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_img, x2), min(h_img, y2)
            crop = img[y1:y2, x1:x2]
            if crop.size > 0 and crop.shape[0] > 10 and crop.shape[1] > 10:
                cv2.imwrite(f"{output_root}/1_human/{file_id}_{i}.png", cv2.resize(crop, IMG_SIZE))
    
    for attempt in range(20):
        if h_img <= 128 or w_img <= 64: break
        ry = np.random.randint(0, h_img - 128)
        rx = np.random.randint(0, w_img - 64)
        if not check_overlap([rx, ry, rx+64, ry+128], human_boxes):
            crop = img[ry:ry+128, rx:rx+64]
            cv2.imwrite(f"{output_root}/0_background/{file_id}_bg.png", cv2.resize(crop, IMG_SIZE))
            break

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known inc

In [ ]:
# Defining model


In [2]:
import torch
import torch.nn as nn 

In [9]:
import torch.nn as nn

class HumanDetector(nn.Module):
    def __init__(self):
        super(HumanDetector, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            
            nn.Dropout(0.5), 
            
            nn.Linear(128 * 16 * 8, 64), 
            nn.ReLU(),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.fc(out)
        return out

In [2]:
# Training

In [10]:
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

model = HumanDetector().to(device)
device = torch.device("mps")

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), # Ensure 3 channels
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.ImageFolder(root='processed_data_64x128', transform=transform)

train_size = int(0.8 * len(dataset))
val_size = val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
        
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "model.pth")

Epoch 1/10 | Loss: 0.5766
Epoch 2/10 | Loss: 0.2793
Epoch 3/10 | Loss: 0.1738
Epoch 4/10 | Loss: 0.1401
Epoch 5/10 | Loss: 0.0997
Epoch 6/10 | Loss: 0.0828
Epoch 7/10 | Loss: 0.0503
Epoch 8/10 | Loss: 0.0351
Epoch 9/10 | Loss: 0.0524
Epoch 10/10 | Loss: 0.0365


In [2]:
# webcam -demo 

In [ ]:
import cv2
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms
from PIL import Image

# 1. SETUP DEVICE
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"

# 2. DEFINE MODEL (Must be exact same structure to load weights)
class HumanDetector(nn.Module):
    def __init__(self):
        super(HumanDetector, self).__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.layer2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5), 
            nn.Linear(128 * 16 * 8, 64), nn.ReLU(), 
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.fc(self.layer3(self.layer2(self.layer1(x))))

# Load Weights
model = HumanDetector().to(device)
model.load_state_dict(torch.load("model.pth"))
model.eval() 

# 3. PREPROCESSING
transform = transforms.Compose([
    transforms.Resize((128, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. START WEBCAM
cap = cv2.VideoCapture(0)
object_detector = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=50)

print("Camera Active! Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    mask = object_detector.apply(frame)
    _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        if cv2.contourArea(cnt) > 2000: # Only look at big moving objects
            x, y, w, h = cv2.boundingRect(cnt)
            
            roi = frame[y:y+h, x:x+w]
            if roi.size == 0: continue
            
            # Convert OpenCV (BGR) -> PIL (RGB) -> Tensor
            pil_img = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            input_tensor = transform(pil_img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                score = model(input_tensor).item()

            if score > 0.8: # Threshold
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 3)
                cv2.putText(frame, f"Human: {score:.2f}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)


    cv2.imshow("Human Detector", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

Camera Active! Press 'q' to quit.


-1

In [ ]:
# testing for the accuracy of the model ( but the model lose where to look, so this is a kind of data leak, but I am doing this just for finding accuracy)

In [ ]:
import cv2
import torch
import torch.nn as nn
import os
import glob
import xml.etree.ElementTree as ET
import numpy as np
from torchvision import transforms
from PIL import Image

# --- CONFIGURATION ---
test_root = "/Users/shudhanshuranjangupta/human/Test"
model_path = "human_detector_m2.pth"

# 1. DEFINE MODEL (Must match training exactly)
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"

class HumanDetector(nn.Module):
    def __init__(self):
        super(HumanDetector, self).__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.layer2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5), 
            nn.Linear(128 * 16 * 8, 64), nn.ReLU(), 
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.fc(self.layer3(self.layer2(self.layer1(x))))

# Load Model
print(f"Loading model from {model_path}...")
model = HumanDetector().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# 2. PREPROCESSING
transform = transforms.Compose([
    transforms.Resize((128, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 3. LOAD FILE LIST
image_paths = sorted(glob.glob(os.path.join(test_root, "JPEGImages", "*.png")))
print(f"Found {len(image_paths)} test images.")

current_idx = 0

def get_ground_truth(xml_path):
    boxes = []
    if not os.path.exists(xml_path): return boxes
    try:
        root = ET.parse(xml_path).getroot()
        for obj in root.findall('object'):
            name = obj.find('name').text.lower()
            if 'person' in name or 'pedestrian' in name:
                bnd = obj.find('bndbox')
                boxes.append([
                    int(float(bnd.find('xmin').text)),
                    int(float(bnd.find('ymin').text)),
                    int(float(bnd.find('xmax').text)),
                    int(float(bnd.find('ymax').text))
                ])
    except: pass
    return boxes

def process_image(idx):
    img_path = image_paths[idx]
    file_id = os.path.splitext(os.path.basename(img_path))[0]
    xml_path = os.path.join(test_root, "Annotations", file_id + ".xml")
    
    # Load Image
    img = cv2.imread(img_path)
    if img is None: return None
    
    # Load Ground Truth (Red Boxes)
    gt_boxes = get_ground_truth(xml_path)
    
    # 1. Draw RED Boxes (Ground Truth)
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2) # Red
        
        # 2. ASK THE MODEL: "Is this a human?"
        # Crop the red box area
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: continue
        
        # Preprocess
        pil_img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        input_tensor = transform(pil_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            score = model(input_tensor).item()
            
        # 3. Draw GREEN Box if Model Agrees (Prediction)
        if score > 0.5:
            # We draw the green box slightly larger/offset so you can see both
            cv2.rectangle(img, (x1-2, y1-2), (x2+2, y2+2), (0, 255, 0), 2) # Green
            label = f"Human: {score:.2f}"
            cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        else:
            # Model Missed it!
            label = f"Miss: {score:.2f}"
            cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # Add UI Text
    cv2.putText(img, f"Image {idx+1}/{len(image_paths)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
    cv2.putText(img, "[A] Prev  [D] Next  [Q] Quit", (10, img.shape[0]-20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    return img

# 4. MAIN LOOP
print("Starting Visualizer...")
while True:
    frame = process_image(current_idx)
    
    if frame is not None:
        cv2.imshow("Test Visualizer", frame)
    
    key = cv2.waitKey(0) # Wait indefinitely for a key press
    
    if key == ord('q'): # Quit
        break
    elif key == ord('d') or key == 83: # Next (d or Right Arrow)
        current_idx = (current_idx + 1) % len(image_paths)
    elif key == ord('a') or key == 81: # Prev (a or Left Arrow)
        current_idx = (current_idx - 1) % len(image_paths)

cv2.destroyAllWindows()
cv2.waitKey(1)

Loading model from human_detector_m2.pth...
Found 288 test images.
Starting Visualizer...


In [3]:
# video_ visualiser

In [ ]:
import cv2
import torch
import torch.nn as nn
import numpy as np
import os
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION ---
input_video_path = "/Users/shudhanshuranjangupta/human/Input_video.mp4" # <--- CHANGE THIS to video filename
output_video_path = "/Users/shudhanshuranjangupta/human/Output_result.mp4"

# 1. STRICTER SETTINGS
CONFIDENCE_THRESHOLD = 0.85  # Only draw if 85% sure
NMS_THRESHOLD = 0.3          # Merge boxes that overlap by 30%
NEW_WIDTH = 960              # Resize for speed

# 2. SETUP MODEL
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"

class HumanDetector(nn.Module):
    def __init__(self):
        super(HumanDetector, self).__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.layer2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5), 
            nn.Linear(128 * 16 * 8, 64), nn.ReLU(), 
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.fc(self.layer3(self.layer2(self.layer1(x))))

model = HumanDetector().to(device)
if os.path.exists("model.pth"):
    model.load_state_dict(torch.load("model.pth", map_location=device))
else:
    raise FileNotFoundError("Model missing!")
model.eval()

# 3. VIDEO PROCESSING
transform = transforms.Compose([
    transforms.Resize((128, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

cap = cv2.VideoCapture(input_video_path)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Resize logic
aspect_ratio = orig_h / orig_w
new_height = int(NEW_WIDTH * aspect_ratio)
print(f"Processing at: {NEW_WIDTH}x{new_height}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter(output_video_path, fourcc, fps, (NEW_WIDTH, new_height))

object_detector = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=50)

for _ in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret: break
    
    # Resize
    frame = cv2.resize(frame, (NEW_WIDTH, new_height))
    
    # Motion Detection
    mask = object_detector.apply(frame)
    _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # --- STEP 1: COLLECT CANDIDATES ---
    boxes = []
    confidences = []

    for cnt in contours:
        if cv2.contourArea(cnt) > 800:
            x, y, w, h = cv2.boundingRect(cnt)
            
            # FILTER 1: Aspect Ratio Check
            # Humans are taller than they are wide.
            # If Width > Height, it's probably a car or a road line.
            if w > h: 
                continue 

            roi = frame[y:y+h, x:x+w]
            if roi.size == 0: continue

            pil_img = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            input_tensor = transform(pil_img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                score = model(input_tensor).item()

            if score > CONFIDENCE_THRESHOLD:
                boxes.append([x, y, w, h])
                confidences.append(float(score))

    # --- STEP 2: APPLY NMS (Non-Maximum Suppression) ---
    # This merges overlapping boxes into one "best" box
    if len(boxes) > 0:
        indices = cv2.dnn.NMSBoxes(boxes, confidences, CONFIDENCE_THRESHOLD, NMS_THRESHOLD)
        
        # --- STEP 3: DRAW ONLY THE WINNERS ---
        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                score = confidences[i]
                
                # Draw Green Box
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                
                # Draw Label
                label = f"Human {score:.2f}"
                cv2.putText(frame, label, (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print(f"\n{output_video_path}")

Processing at: 960x540


100%|█████████████████████████████████████████| 264/264 [00:10<00:00, 24.99it/s]


✅ Done! Check file: /Users/shudhanshuranjangupta/human/output_result.mp4


In [4]:
# Image visualiser

In [ ]:
import cv2
import torch
import torch.nn as nn
import numpy as np
import os
from torchvision import transforms
from PIL import Image

# --- CONFIGURATION ---
image_path = "/Users/shudhanshuranjangupta/human/Input_image.png" 

# TUNING
SCALES = [400, 600, 800, 1000] # Added more scales to catch him
WINDOW_SIZE = (64, 128)
STEP_SIZE = 16 
CONF_THRESH = 0.80     # Lowered slightly to catch the man
HEATMAP_THRESH = 3     

# 1. SETUP MODEL
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"

class HumanDetector(nn.Module):
    def __init__(self):
        super(HumanDetector, self).__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2))
        self.layer2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5), 
            nn.Linear(128 * 16 * 8, 64), nn.ReLU(), 
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.fc(self.layer3(self.layer2(self.layer1(x))))

model = HumanDetector().to(device)
if os.path.exists("human_detector_m2.pth"):
    model.load_state_dict(torch.load("human_detector_m2.pth", map_location=device))
    model.eval()
else:
    print("Error: Model file missing.")

transform = transforms.Compose([
    transforms.Resize((128, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def sliding_window(image, step, window_size):
    for y in range(0, image.shape[0] - window_size[1], step):
        for x in range(0, image.shape[1] - window_size[0], step):
            yield (x, y, image[y:y + window_size[1], x:x + window_size[0]])

# 3. MAIN LOGIC
if not os.path.exists(image_path):
    print(f"Error: File not found at {image_path}")
else:
    original_img = cv2.imread(image_path)
    
    if original_img is None:
        print(f"Error: Cannot read image at {image_path}")
    else:
        orig_h, orig_w = original_img.shape[:2]
        master_heatmap = np.zeros((orig_h, orig_w), dtype=np.float32)

        print(f"Scanning Image")

        for target_width in SCALES:
            aspect_ratio = orig_h / orig_w
            target_height = int(target_width * aspect_ratio)
            img_scaled = cv2.resize(original_img, (target_width, target_height))
            
            ratio_x = orig_w / target_width
            ratio_y = orig_h / target_height
            
            for (x, y, window) in sliding_window(img_scaled, STEP_SIZE, WINDOW_SIZE):
                if window.shape[0] != WINDOW_SIZE[1] or window.shape[1] != WINDOW_SIZE[0]: continue

                pil_img = Image.fromarray(cv2.cvtColor(window, cv2.COLOR_BGR2RGB))
                input_tensor = transform(pil_img).unsqueeze(0).to(device)
                
                with torch.no_grad():
                    score = model(input_tensor).item()

                if score > CONF_THRESH:
                    real_x = int(x * ratio_x)
                    real_y = int(y * ratio_y)
                    real_w = int(WINDOW_SIZE[0] * ratio_x)
                    real_h = int(WINDOW_SIZE[1] * ratio_y)
                    
                    # DEBUG: Draw faint blue box for EVERY detection (Raw View)
                    cv2.rectangle(original_img, (real_x, real_y), (real_x+real_w, real_y+real_h), (255, 0, 0), 1)

                    y2 = min(real_y + real_h, orig_h)
                    x2 = min(real_x + real_w, orig_w)
                    master_heatmap[real_y:y2, real_x:x2] += 1

        print("Filtering Results...")
        master_heatmap[master_heatmap < HEATMAP_THRESH] = 0
        master_heatmap[master_heatmap >= HEATMAP_THRESH] = 255
        master_heatmap = master_heatmap.astype(np.uint8)

        contours, _ = cv2.findContours(master_heatmap, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        count = 0
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            
            # --- STRICT GEOMETRY FILTERS ---
            # 1. Aspect Ratio: Must be TALLER than WIDE (Fixes the street box)
            if w > h: 
                continue 
            
            # 2. Too Big: Humans usually aren't 50% of the screen width
            if w > (orig_w * 0.5):
                continue
                
            # 3. Too Small: Ignore tiny noise
            if h < (orig_h * 0.05):
                continue

            # Draw FINAL Green Box
            cv2.rectangle(original_img, (x, y), (x+w, y+h), (0, 255, 0), 4)
            cv2.putText(original_img, "Human", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            count += 1

        print(f"Final Count: {count}")
        
        # Display
        display_h = 800
        display_w = int(display_h * (orig_w / orig_h))
        preview = cv2.resize(original_img, (display_w, display_h))
        
        cv2.imshow("Debug Result (Blue=Raw, Green=Final)", preview)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
        cv2.waitKey(1)

Scanning Image...
Filtering Results...
Final Count: 2
